# 1. Cấu hình và Load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta, datetime

# CẤU HÌNH ĐƯỜNG DẪN (QUAN TRỌNG) 
current_folder = os.getcwd() 
DATA_DIR = os.path.join(current_folder, 'dataset\Input (BTC)')

# Setup seed
np.random.seed(42)

print(f"📂 Đang đọc dữ liệu từ: {DATA_DIR}")

# 2. Load các file CSV 
try:
    df_products = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
    df_stores = pd.read_csv(os.path.join(DATA_DIR, 'stores.csv'))
    df_trans = pd.read_csv(os.path.join(DATA_DIR, 'transactions.csv')) 
    df_customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
    if os.path.exists(os.path.join(DATA_DIR, 'discounts.csv')):
        df_discounts = pd.read_csv(os.path.join(DATA_DIR, 'discounts.csv'))
    else:
        df_discounts = pd.DataFrame()
        
    print("✅ Load dữ liệu thành công!")
    print(f"- Số sản phẩm: {df_products.shape[0]}")
    print(f"- Số giao dịch: {df_trans.shape[0]}")
    
except FileNotFoundError as e:
    print("❌ LỖI: Không tìm thấy file!")
    print(f"Đường dẫn đang tìm: {DATA_DIR}")

📂 Đang đọc dữ liệu từ: c:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Input (BTC)
✅ Load dữ liệu thành công!
- Số sản phẩm: 17940
- Số giao dịch: 6416827


C:\Users\Dell\AppData\Local\Temp\ipykernel_2900\625836177.py:26: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))


# 2. Chuyển đổi sang brand Việt Nam

In [10]:
# Danh sách Brand muốn demo
target_brands = ['Routine', 'Coolmate', 'H&M', 'Zara', 'DirtyCoins']

# 1. Lọc lấy sản phẩm là Áo (Tops/T-shirts)
# Giả định lọc theo Category hoặc Sub Category
mask_shirt = df_products['Sub Category'].astype(str).str.contains('T-shirt|Top|Shirt', case=False, na=False)
hero_products = df_products[mask_shirt].copy()

# Nếu ít quá thì lấy 50 sản phẩm đầu tiên
if len(hero_products) < 10:
    hero_products = df_products.head(50).copy()
else:
    hero_products = hero_products.head(50).reset_index(drop=True)

# 2. Hàm đổi tên và gán Brand
def localize_data(row):
    brand = np.random.choice(target_brands)
    # Tạo tên sản phẩm giả: Brand + Màu + Loại
    color = str(row['Color']) if pd.notna(row['Color']) else "Basic"
    new_name = f"{brand} {color} Streetwear Tee {np.random.randint(100,999)}"
    
    # Random giá bán tại VN (từ 200k đến 1000k)
    vn_price = np.random.choice([249000, 299000, 349000, 399000, 499000, 599000, 699000, 799000, 899000, 999000])
    
    return pd.Series([brand, new_name, vn_price])

# Áp dụng hàm
hero_products[['Brand_VN', 'Name_VN', 'Price_VND']] = hero_products.apply(localize_data, axis=1)

# 3. Tạo cột trống để điền link ảnh sau này
hero_products['image_url'] = ''      # Link ảnh web (hiện trong AR Catalog)
hero_products['model_3d_url'] = ''   # Link file 3D hoặc ảnh tách nền (hiện trong AR Camera)

# 4. Xuất file để team làm content
output_prod_path = os.path.join(DATA_DIR, 'processed_products_VN.csv')
hero_products.to_csv(output_prod_path, index=False)

print(f"Đã tạo file danh sách sản phẩm: {output_prod_path}")


Đã tạo file danh sách sản phẩm: c:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Input (BTC)\processed_products_VN.csv


- print("👉 Task: Mở file này lên, chọn 5-10 dòng, điền link ảnh thật vào cột 'image_url'.")

# 3. Xử lý Transaction (Data cho Forecast)

- Gom nhóm giao dịch theo ngày để chạy mô hình dự báo

In [11]:
# 1. Xử lý ngày tháng
df_trans['Date'] = pd.to_datetime(df_trans['Date'])

# 2. Lọc giao dịch chỉ của các sản phẩm Hero 
hero_ids = hero_products['Product ID'].unique()
df_trans_filtered = df_trans[df_trans['Product ID'].isin(hero_ids)].copy()

# 3. Resample theo Ngày (Daily Sales)
# Group by: Ngày, Cửa hàng, Sản phẩm -> Tính tổng số lượng và doanh thu
daily_sales = df_trans_filtered.groupby([
    pd.Grouper(key='Date', freq='D'), 
    'Store ID', 
    'Product ID'
]).agg({
    'Quantity': 'sum',
    'Line Total': 'sum'
}).reset_index()

daily_sales.rename(columns={'Line Total': 'Revenue', 'Date': 'Sales_Date'}, inplace=True)

# 4. Feature Engineering 
# Thêm cột Thứ trong tuần (0=Thứ 2, 6=CN)
daily_sales['day_of_week'] = daily_sales['Sales_Date'].dt.dayofweek
# Thêm cột Tháng
daily_sales['month'] = daily_sales['Sales_Date'].dt.month
# Thêm cột Cuối tuần (True/False)
daily_sales['is_weekend'] = daily_sales['day_of_week'] >= 5

# Lưu file training
output_train_path = os.path.join(DATA_DIR, 'train_data_forecast.csv')
daily_sales.to_csv(output_train_path, index=False)
print(f"Đã tạo dữ liệu train Forecast: {output_train_path}")

Đã tạo dữ liệu train Forecast: c:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Input (BTC)\train_data_forecast.csv


# 4. Mockup Lịch sử giá

- Tạo dữ liệu giả lập biến động giá

In [12]:
price_history_data = []

print("Đang tạo dữ liệu lịch sử giá cho giai đoạn 2024-2025...")

for pid in hero_ids:
    # Lấy giá gốc VND hiện tại
    base_price = hero_products[hero_products['Product ID'] == pid]['Price_VND'].values[0]
    
    # --- NĂM 2024 ---
    
    # 1. Giá ổn định đầu năm 2024 (01/01 - 14/11)
    price_history_data.append({
        'Product ID': pid,
        'Price': base_price,
        'Start_Date': '2024-01-01',
        'End_Date': '2024-11-14',
        'Is_Promo': 0,
        'Event': 'Standard 2024'
    })
    
    # 2. Black Friday 2024 (15/11 - 30/11) -> Giảm 20-30%
    bf_discount = np.random.choice([0.7, 0.8]) # Giá còn 70% hoặc 80%
    price_history_data.append({
        'Product ID': pid,
        'Price': int(base_price * bf_discount),
        'Start_Date': '2024-11-15',
        'End_Date': '2024-11-30',
        'Is_Promo': 1,
        'Event': 'Black Friday 2024'
    })
    
    # 3. Cuối năm & Trước Tết (01/12/2024 - 10/01/2025) -> Giá về gốc
    price_history_data.append({
        'Product ID': pid,
        'Price': base_price,
        'Start_Date': '2024-12-01',
        'End_Date': '2025-01-10',
        'Is_Promo': 0,
        'Event': 'Pre-Tet'
    })

    # --- NĂM 2025 ---

    # 4. Sale Tết Ất Tỵ 2025 (11/01 - 31/01) -> Giảm 10-15%
    tet_discount = np.random.choice([0.85, 0.9])
    price_history_data.append({
        'Product ID': pid,
        'Price': int(base_price * tet_discount),
        'Start_Date': '2025-01-11',
        'End_Date': '2025-01-31',
        'Is_Promo': 1,
        'Event': 'Tet Sale 2025'
    })
    
    # 5. Sau Tết & Đầu Hè (01/02 - 31/05) -> Giá gốc
    price_history_data.append({
        'Product ID': pid,
        'Price': base_price,
        'Start_Date': '2025-02-01',
        'End_Date': '2025-05-31',
        'Is_Promo': 0,
        'Event': 'Standard Spring 2025'
    })
    
    # 6. Summer Sale 2025 (01/06 - 15/07) -> Giảm 10%
    price_history_data.append({
        'Product ID': pid,
        'Price': int(base_price * 0.9),
        'Start_Date': '2025-06-01',
        'End_Date': '2025-07-15',
        'Is_Promo': 1,
        'Event': 'Summer Sale 2025'
    })
    
    # 7. Giai đoạn hiện tại & Cuối năm (16/07 - 15/12/2025)
    # Giả lập tăng nhẹ giá 5% do lạm phát/chi phí vận hành cuối năm
    new_base_price = int(base_price * 1.05)
    price_history_data.append({
        'Product ID': pid,
        'Price': new_base_price,
        'Start_Date': '2025-07-16',
        'End_Date': '2025-12-15',
        'Is_Promo': 0,
        'Event': 'Standard End 2025'
    })

df_price_hist = pd.DataFrame(price_history_data)

# Lưu file
output_price_path = os.path.join(DATA_DIR, 'fact_price_history.csv')
df_price_hist.to_csv(output_price_path, index=False)

print(f"Đã tạo bảng lịch sử giá 2024-2025: {output_price_path}")
print("Mẫu dữ liệu (5 dòng đầu):")
print(df_price_hist[['Product ID', 'Start_Date', 'Price', 'Event']].head())

Đang tạo dữ liệu lịch sử giá cho giai đoạn 2024-2025...
Đã tạo bảng lịch sử giá 2024-2025: c:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Input (BTC)\fact_price_history.csv
Mẫu dữ liệu (5 dòng đầu):
   Product ID  Start_Date   Price                 Event
0           4  2024-01-01  799000         Standard 2024
1           4  2024-11-15  559300     Black Friday 2024
2           4  2024-12-01  799000               Pre-Tet
3           4  2025-01-11  719100         Tet Sale 2025
4           4  2025-02-01  799000  Standard Spring 2025


# 5. Mockup Tương tác (Fact Interactions)

- Tạo dữ liệu hành vi người dùng (View, Try-on) dựa trên lịch sử mua hàng.

In [13]:
interactions = []
# Các loại hành động trên App Sentio
actions = ['VIEW_AR_CATALOG', 'TRY_ON_VIRTUAL', 'SCAN_STOREFRONT', 'ADD_TO_CART']

print("Đang giả lập hành vi người dùng (Interactions)...")

# Duyệt qua các giao dịch thật để sinh ra tương tác ảo tương ứng
# Lưu ý: Tương tác này sẽ xảy ra trước thời điểm mua hàng (trong quá khứ)
for idx, row in df_trans_filtered.iterrows():
    # Lấy thông tin cơ bản
    p_id = row['Product ID']
    c_id = row['Customer ID']
    buy_date = row['Date'] # Ngày mua hàng thực tế
    
    # Sinh ra 3-5 tương tác trước khi mua
    num_actions = np.random.randint(3, 6)
    
    for _ in range(num_actions):
        # Random ngày tương tác (trong vòng 7 ngày trước khi mua)
        lag = np.random.randint(0, 7)
        inter_date = buy_date - timedelta(days=lag, minutes=np.random.randint(10, 300))
        
        interactions.append({
            'Interaction_ID': f"INT_{np.random.randint(100000, 999999)}",
            'Customer_ID': c_id,
            'Product_ID': p_id,
            'Action_Type': np.random.choice(actions, p=[0.4, 0.3, 0.2, 0.1]), # Tỷ lệ: AR View nhiều nhất
            'Timestamp': inter_date,
            'Duration_Sec': np.random.randint(5, 120) # Xem trong bao nhiêu giây
        })

df_interactions = pd.DataFrame(interactions)
output_inter_path = os.path.join(DATA_DIR, 'fact_user_interactions.csv')
df_interactions.to_csv(output_inter_path, index=False)

print(f"Đã tạo bảng tương tác người dùng: {output_inter_path}")
print(f"- Tổng số lượt tương tác: {len(df_interactions)}")
print("- Mẫu dữ liệu:")
print(df_interactions.head())

Đang giả lập hành vi người dùng (Interactions)...
Đã tạo bảng tương tác người dùng: c:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Input (BTC)\fact_user_interactions.csv
- Tổng số lượt tương tác: 24430
- Mẫu dữ liệu:
  Interaction_ID  Customer_ID  Product_ID      Action_Type  \
0     INT_468452        31756         117   TRY_ON_VIRTUAL   
1     INT_122671        31756         117  SCAN_STOREFRONT   
2     INT_411955        31756         117  VIEW_AR_CATALOG   
3     INT_242483        38375         239   TRY_ON_VIRTUAL   
4     INT_843974        38375         239      ADD_TO_CART   

            Timestamp  Duration_Sec  
0 2022-12-26 08:33:00            74  
1 2022-12-27 09:13:00            63  
2 2022-12-27 08:54:00            40  
3 2022-12-31 06:25:00            56  
4 2023-01-01 06:44:00             5  


# 6. Biến đổi và xác định sản phẩm demo

In [ ]:
import pandas as pd
import os

file_path = r"C:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Output (Sentio)\processed_products_VN.csv"

try:
    df = pd.read_csv(file_path)
    
    # 1. DROP CÁC CỘT KHÔNG CẦN THIẾT
    cols_to_drop = ['Description PT', 'Description DE', 'Description FR', 'Description ES', 'Description ZH']
    df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    
    # Đổi tên Description EN -> Description
    if 'Description EN' in df.columns:
        df.rename(columns={'Description EN': 'Description'}, inplace=True)

    # 2. CHUẨN BỊ DATA DEMO CHO 5 BRAND (Mỗi brand 2 sản phẩm)
    demo_descriptions = {
        'Routine': [
            "Áo dệt kim nam tay ngắn vặn thừng mang phong cách lịch lãm pha nét hiện đại, nổi bật với họa tiết vặn thừng tinh tế tạo chiều sâu và điểm nhấn cho trang phục. Thiết kế tay ngắn gọn gàng, phù hợp cho cả đi làm, dạo phố hay những dịp cần vẻ ngoài chỉn chu nhưng không quá trang trọng.",
            "Mẫu áo polo tay ngắn nữ là sản phẩm với kiểu dáng thời thượng và chất liệu cao cấp, mang đến vẻ đẹp tinh tế và cuốn hút cho người mặc"
        ],
        'Coolmate': [
            "Một chiếc áo polo nam tiên phong được tạo nên từ sự kết hợp độc đáo giữa sợi S.Café® (từ bã cà phê) và sợi PET tái chế (từ chai nhựa).",
            "Thiết kế áo chạy bộ nổi bật với hiệu ứng chuyển màu tinh tế không chỉ tạo nên dấu ấn thị giác mạnh mẽ mà còn thể hiện tinh thần không ngừng vươn xa. Hiệu suất tối đa đến từ chất liệu 100% vải polyester siêu nhẹ, kết hợp cùng công nghệ Ex-Dry tiên tiến."
        ],
        'H&M': [
            "Áo ôm, dài tay dệt kim gân nổi mềm làm từ viscose pha có chứa sợi len. Cổ chữ V không đối xứng có chi tiết nhún vải ở một bên tạo hiệu ứng xếp rủ.",
            "Áo thun bằng jersey nhẹ, in hình làm từ cotton pha. Cổ tròn, viền gân nổi và vạt ngang"
        ],
        'Zara': [
            "Áo phông dệt kim dáng suông được làm từ sợi pha lyocell và cotton. Cổ tròn, cộc tay. Bo viền bằng vải gân",
            "Áo sơ mi làm từ sợi viscose. Cổ tròn và cổ chữ V khoét sâu có dây buộc điều chỉnh. Tay dài loe. Chi tiết bèo nhún cùng chất liệu. Cài khuy phía trước."
        ],
        'DirtyCoins': [
            "Phối màu Vàng - Xanh với hiệu ứng loang màu độc bản, không đụng hàng. Mặt sau in số 9 Big Size cùng logo DirtyCoins – một lời khẳng định cái tôi đầy kiêu hãnh.",
            "Chiếc Sweater màu Baby Pink này chính là minh chứng cho việc bạn có thể vừa keo lì vừa cực kỳ đáng yêu"
        ]
    }

    # 3. CẬP NHẬT MÔ TẢ VÀO DATAFRAME
    for brand, descs in demo_descriptions.items():
        # Tìm các dòng thuộc Brand này
        brand_rows = df[df['Brand_VN'] == brand]
        
        # Nếu tìm thấy sản phẩm của Brand đó
        if not brand_rows.empty:
            # Lấy tối đa 2 sản phẩm đầu tiên để sửa
            indices_to_update = brand_rows.index[:2]
            
            for i, idx in enumerate(indices_to_update):
                # Cập nhật mô tả (Lấy mô tả tương ứng, nếu hết thì lấy cái cuối)
                desc_text = descs[i] if i < len(descs) else descs[-1]
                
                df.at[idx, 'Description'] = desc_text
                # Đánh dấu đây là hàng Demo để dễ tìm
                df.at[idx, 'Is_Demo'] = 1 
                
                print(f"    Đã sửa: {brand} - ID {df.at[idx, 'Product ID']}")

    # 4. LƯU FILE MỚI
    output_dir = os.path.dirname(file_path)
    output_path = os.path.join(output_dir, 'processed_products_VN_final.csv')
    df.to_csv(output_path, index=False)
    
    print("-" * 30)
    print(f"XONG! File mới đã lưu tại: {output_path}")
    
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại '{file_path}'. Kiểm tra lại xem file csv đang để ở đâu.")

    Đã sửa: Routine - ID 40
    Đã sửa: Routine - ID 56
    Đã sửa: Coolmate - ID 13
    Đã sửa: Coolmate - ID 31
    Đã sửa: H&M - ID 14
    Đã sửa: H&M - ID 30
    Đã sửa: Zara - ID 4
    Đã sửa: Zara - ID 57
    Đã sửa: DirtyCoins - ID 5
    Đã sửa: DirtyCoins - ID 15
------------------------------
XONG! File mới đã lưu tại: C:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Output (Sentio)\processed_products_VN_final.csv


# 7. Kiểm tra ảnh và dữ liệu demo có khớp nhau hay không

In [17]:
import pandas as pd
import os

csv_path = r"C:\Users\Dell\Downloads\Sentio_Datastorm_2025\dataset\Output (Sentio)\processed_products_VN_final.csv"

# Đường dẫn đến folder chứa ảnh thật
images_folder = r"C:\Users\Dell\Downloads\Sentio_Datastorm_2025\media\tryon\garments"

def run_sanity_check():
    
    # Bước 1: Kiểm tra xem file và folder có tồn tại không
    if not os.path.exists(csv_path):
        print(f"LỖI: Không tìm thấy file CSV tại: {csv_path}")
        return
    if not os.path.exists(images_folder):
        print(f"LỖI: Không tìm thấy folder ảnh tại: {images_folder}")
        return

    # Bước 2: Đọc file CSV
    try:
        df = pd.read_csv(csv_path)
        print(f"Đã đọc file CSV: {len(df)} dòng sản phẩm.")
    except Exception as e:
        print(f"Lỗi đọc file CSV: {e}")
        return

    # Bước 3: Lấy danh sách ảnh trong folder
    # Lấy tất cả file trong folder và chuyển về chữ thường để so sánh cho dễ
    actual_files = set(os.listdir(images_folder))
    print(f"Đã quét folder ảnh: Tìm thấy {len(actual_files)} file.\n")

    # Bước 4: So sánh từng dòng
    missing_images = []
    valid_count = 0
    demo_count = 0

    print("-" * 50)
    print(f"{'SẢN PHẨM':<40} | {'TRẠNG THÁI':<15} | {'TÊN FILE ẢNH'}")
    print("-" * 50)

    for index, row in df.iterrows():
        # Chỉ kiểm tra những dòng có điền ảnh (không bị rỗng/NaN)
        img_name = str(row['image_url']).strip()
        
        if img_name != 'nan' and img_name != '':
            demo_count += 1
            
            # Kiểm tra xem file có nằm trong folder không
            if img_name in actual_files:
                print(f"{row['Name_VN'][:38]:<40} | ✅ OK          | {img_name}")
                valid_count += 1
            else:
                print(f"{row['Name_VN'][:38]:<40} | ❌ MISSING     | {img_name}")
                missing_images.append((row['Product ID'], row['Name_VN'], img_name))
    
    print("-" * 50)
    
    # Báo cáo tổng kết
    print("\n BÁO CÁO TỔNG KẾT:")
    print(f"- Tổng số sản phẩm cần Demo (có điền link): {demo_count}")
    print(f"- Số ảnh HỢP LỆ (Tìm thấy): {valid_count}")
    print(f"- Số ảnh LỖI (Không tìm thấy): {len(missing_images)}")

    if len(missing_images) > 0:
        print("\n CẢNH BÁO: Các sản phẩm sau đây sẽ KHÔNG hiện ảnh:")
        for pid, name, img in missing_images:
            print(f"   • ID {pid}: {name} -> File '{img}' không có trong folder!")
        print("\n Gợi ý sửa lỗi:")
        print("   1. Kiểm tra lại xem đã copy ảnh vào folder 'garments' chưa?")
        print("   2. Kiểm tra chính tả tên file (ví dụ: '.jpg' khác '.jpeg', chữ hoa/thường).")
    else:
        if demo_count > 0:
            print("\n TUYỆT VỜI! Mọi thứ đã khớp 100%. Sẵn sàng Demo!")
        else:
            print("\n CHÚ Ý: Chưa điền tên ảnh nào vào file CSV cả. Hãy điền ít nhất 1 cái!")

# Chạy hàm
run_sanity_check()

Đã đọc file CSV: 50 dòng sản phẩm.
Đã quét folder ảnh: Tìm thấy 10 file.

--------------------------------------------------
SẢN PHẨM                                 | TRẠNG THÁI      | TÊN FILE ẢNH
--------------------------------------------------
Zara Basic Streetwear Tee 960            | ✅ OK          | Zara_do.jpg
DirtyCoins Basic Streetwear Tee 120      | ✅ OK          | Dirtycoins_xanh.jpg
Coolmate RED Streetwear Tee 566          | ✅ OK          | Coolmate_den.jpg
H&M Basic Streetwear Tee 558             | ✅ OK          | H&M_nau.jpg
DirtyCoins Basic Streetwear Tee 199      | ✅ OK          | Dirtycoins_hong.jpg
H&M Basic Streetwear Tee 761             | ✅ OK          | H&M_xam.jpg
Coolmate Basic Streetwear Tee 443        | ✅ OK          | Coolmate_hong.jpg
Routine GOLD Streetwear Tee 559          | ✅ OK          | Routine_do.jpg
Routine Basic Streetwear Tee 574         | ✅ OK          | Routine_xanh.jpg
Zara Basic Streetwear Tee 799            | ✅ OK          | Zara_vang.jpg
---

# 8. Tạo khung dữ liệu năm 2023 (Clone & Shift)

In [ ]:
# 1. Lấy dữ liệu năm 2024 làm mẫu
df_2024 = df[df['Start_Date'].dt.year == 2024].copy()

# 2. Hàm lùi ngày lại 1 năm (xử lý cả năm nhuận)
def shift_date_back_1_year(date):
    try:
        return date.replace(year=date.year - 1)
    except ValueError:
        # Xử lý ngày 29/2 của năm nhuận -> về 28/2
        return date.replace(month=2, day=28, year=date.year - 1)

# 3. Tạo DataFrame cho năm 2023
df_2023 = df_2024.copy()

# Áp dụng lùi ngày
df_2023['Start_Date'] = df_2023['Start_Date'].apply(shift_date_back_1_year)
df_2023['End_Date'] = df_2023['End_Date'].apply(shift_date_back_1_year)

# 4. Đổi tên sự kiện (Thay thế "2024" thành "2023")
df_2023['Event'] = df_2023['Event'].str.replace('2024', '2023', regex=False)

print(f"Đã tạo khung dữ liệu sơ bộ cho 2023: {len(df_2023)} dòng.")

# 9. Biến động giá

- Áp dụng logic: Giá 2023 rẻ hơn 2024 (do lạm phát) và Thêm chút noise vào giá khuyến mãi để không trùng khớp 100%

In [ ]:
# CẤU HÌNH BIẾN ĐỘNG 
INFLATION_RATE_MIN = 0.90  # Giá 2023 bằng 90% giá 2024
INFLATION_RATE_MAX = 0.96  # Giá 2023 bằng 96% giá 2024
PROMO_NOISE_RANGE = 0.02   # Biến động +/- 2% cho các đợt Sale

# Hàm làm tròn giá 
def round_price(price):
    return int(round(price / 1000.0) * 1000)

# 1. XỬ LÝ GIÁ GỐC (Is_Promo = 0)
# Giả lập lạm phát: Giá năm ngoái rẻ hơn năm nay một chút
mask_normal = df_2023['Is_Promo'] == 0
inflation_factors = np.random.uniform(INFLATION_RATE_MIN, INFLATION_RATE_MAX, size=mask_normal.sum())

# Áp dụng giảm giá và làm tròn
df_2023.loc[mask_normal, 'Price'] = (df_2023.loc[mask_normal, 'Price'] * inflation_factors).astype(int)
df_2023.loc[mask_normal, 'Price'] = df_2023.loc[mask_normal, 'Price'].apply(round_price)

# 2. XỬ LÝ GIÁ KHUYẾN MÃI (Is_Promo = 1)
# Thêm độ nhiễu ngẫu nhiên (Noise) để các đợt sale trông tự nhiên hơn
mask_promo = df_2023['Is_Promo'] == 1
noise_factors = np.random.uniform(1 - PROMO_NOISE_RANGE, 1 + PROMO_NOISE_RANGE, size=mask_promo.sum())

# Áp dụng noise và làm tròn
df_2023.loc[mask_promo, 'Price'] = (df_2023.loc[mask_promo, 'Price'] * noise_factors).astype(int)
df_2023.loc[mask_promo, 'Price'] = df_2023.loc[mask_promo, 'Price'].apply(round_price)

print("Đã áp dụng biến động giá thành công!")
print(df_2023[['Product ID', 'Price', 'Event', 'Start_Date']].head())

- Áp dụng logic: "Lạm phát ngược dòng thời gian" (Reverse Inflation)

In [ ]:
# TẠO DỮ LIỆU NĂM 2022 TỪ DỮ LIỆU NĂM 2023
df_2022 = df_2023.copy()

# LÙI NGÀY VỀ 1 NĂM (2023 -> 2022)
# Tái sử dụng hàm lùi ngày đã định nghĩa ở trên
df_2022['Start_Date'] = df_2022['Start_Date'].apply(shift_date_back_1_year)
df_2022['End_Date'] = df_2022['End_Date'].apply(shift_date_back_1_year)

# ĐỔI TÊN SỰ KIỆN ("2023" -> "2022")
df_2022['Event'] = df_2022['Event'].str.replace('2023', '2022', regex=False)

# ÁP DỤNG LOGIC GIÁ (2022 RẺ HƠN 2023)
    
# --- Yếu tố 1: Giá gốc rẻ hơn (Lạm phát) ---
# Giá 2022 = 92% đến 97% giá 2023
INFLATION_FACTOR_22 = np.random.uniform(0.92, 0.97, size=len(df_2022))
    
mask_normal_22 = df_2022['Is_Promo'] == 0
df_2022.loc[mask_normal_22, 'Price'] = (df_2022.loc[mask_normal_22, 'Price'] * INFLATION_FACTOR_22[mask_normal_22]).astype(int)
    
# Làm tròn giá cho đẹp
df_2022['Price'] = df_2022['Price'].apply(round_price)

# --- Yếu tố 2: Giá Sale biến động khác đi (Noise) ---
# Sale năm 2022 sẽ không giảm y hệt Sale năm 2023 (biến động +/- 2%)
mask_promo_22 = df_2022['Is_Promo'] == 1
noise_22 = np.random.uniform(0.98, 1.02, size=mask_promo_22.sum())
    
df_2022.loc[mask_promo_22, 'Price'] = (df_2022.loc[mask_promo_22, 'Price'] * noise_22).astype(int)
df_2022.loc[mask_promo_22, 'Price'] = df_2022.loc[mask_promo_22, 'Price'].apply(round_price)

print(f"Đã tạo xong dữ liệu 2022: {len(df_2022)} dòng.")
print("Ví dụ 5 dòng đầu tiên của 2022:")
print(df_2022[['Product ID', 'Price', 'Event', 'Start_Date']].head())

## Xuất file

In [ ]:
# 1. Gộp với dữ liệu gốc (đã có 2024, 2025)
df_final = pd.concat([df, df_2022, df_2023], ignore_index=True)

# 2. Sắp xếp lại cho dễ nhìn (Theo Sản phẩm -> Thời gian)
df_final = df_final.sort_values(by=['Product ID', 'Start_Date'])

# 3. Kiểm tra kết quả cuối cùng
print("Phân bố dữ liệu sau khi gộp:")
print(df_final['Start_Date'].dt.year.value_counts().sort_index())

# 4. Xuất file CSV
output_filename = 'fact_price_history_full.csv'
df_final.to_csv(output_filename, index=False)
print(f"Đã xuất file thành công: {output_filename}")